# Customer Lifetime Value Analysis

Customer lifetime value (CLTV) is the estimated total amount a customer will spend on a business throughout their relationship with that business. It takes into account the revenue generated by the customer as well as the costs associated with acquiring and serving that customer. By analyzing the relationship between customer acquisition costs and revenue generated, we can determine which channels are the most cost-effective for acquiring and retaining high-value customers.

The given data includes information about the customer’s `channel`, `customer_ids`, `acquisition_channels`, `acquisition_cost`, `conversion rate, and `revenue` generated. 

The task is to analyze the CLTV of customers across different channels and identify the most profitable channels for the business.

In [2]:
# import necessary libraries

import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "plotly_white"

In [3]:
# load dataset into a pandas DataFrame
data = pd.read_csv('customer_acquisition_data.csv')
# check the first few rows of the dataset
print(data.head())

  CustomerID AcquisitionChannel  AcquisitionCost  ConversionRate    Revenue
0      CID_1        Paid Search        48.286408        0.044949  85.780692
1      CID_2          Affiliate        43.758214        0.040348  97.686758
2      CID_3    Email Marketing        11.967483        0.063956  97.383527
3      CID_4       Social Media        28.647234        0.016970  73.356294
4      CID_5            Organic         0.000000        0.060899  44.198727


In [4]:
print(data.shape)
data.info()

(4000, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CustomerID          4000 non-null   object 
 1   AcquisitionChannel  4000 non-null   object 
 2   AcquisitionCost     4000 non-null   float64
 3   ConversionRate      4000 non-null   float64
 4   Revenue             4000 non-null   float64
dtypes: float64(3), object(2)
memory usage: 156.4+ KB


There are 4000 customers in this dataset.

Let's delve deeper into examining the data. We will check for missing data, and examine unique values in `AcquisitionChannel` column

In [5]:
# Check for missing values in each column
print("\n### Missing Values ###")
print(data.isnull().sum())


### Missing Values ###
CustomerID            0
AcquisitionChannel    0
AcquisitionCost       0
ConversionRate        0
Revenue               0
dtype: int64


In [6]:
# Check unique values in the 'AcquisitionChannel' column
print("\n### Value Counts for Acquisition Channel (Percentage) ###")
print(data['AcquisitionChannel'].value_counts(normalize=True) * 100)


### Value Counts for Acquisition Channel (Percentage) ###
AcquisitionChannel
Organic            25.650
Email Marketing    20.250
Paid Search        19.575
Social Media       14.425
Referral           10.450
Affiliate           9.650
Name: proportion, dtype: float64


* **Organic** acquisition is the most significant channel, accounting for over a quarter of all customers (25.65%). This suggests that efforts to improve organic visibility (like SEO, content marketing, or brand awareness) are important.
* **Email Marketing** is also a strong channel, responsible for acquiring a substantial portion of the customer base (20.25%). This highlights the importance of email list building and effective email campaigns.
* **Affiliate** is the smallest channel, contributing the least to customer acquisition (9.65%). This might warrant a closer look at the performance and ROI of the affiliate program.

## Problem Statement 1
**What is the average initial revenue generated by customers acquired through each channel?**

In [7]:
# 1. Average Initial Revenue per Channel
average_revenue_by_channel = data.groupby('AcquisitionChannel')['Revenue'].mean().reset_index()

print("Average Initial Revenue per Channel:")
print(average_revenue_by_channel)
# Visualize the average revenue per channel, sorted by revenue in descending order
fig = px.bar(average_revenue_by_channel,
             x=average_revenue_by_channel.sort_values('Revenue', ascending=False)['AcquisitionChannel'],  # Sort x-axis by Revenue in descending order
             y=average_revenue_by_channel.sort_values('Revenue', ascending=False)['Revenue'],  # Ensure y-axis matches the sorted order
             title='Average Initial Revenue per Acquisition Channel',
             labels={'Revenue': 'Average Revenue', 'AcquisitionChannel': 'Acquisition Channel'},
             color='AcquisitionChannel')  # Add color to differentiate channels
fig.update_layout(
    xaxis_title="Acquisition Channel",
    yaxis_title="Average Revenue"
)

fig.show()

Average Initial Revenue per Channel:
  AcquisitionChannel     Revenue
0          Affiliate  112.197558
1    Email Marketing  149.011730
2            Organic   80.561783
3        Paid Search  119.698917
4           Referral  182.300536
5       Social Media   89.532455


**Based on the chart:**
- Referral has the highest average initial revenue, significantly outperforming the other channels.
- Organic has the lowest average initial revenue. 
- Email Marketing and Paid Search also show relatively strong performance in terms of initial revenue.

This visualization helps us understand which acquisition channels tend to bring in customers with higher initial spending.

## Problem Statement 2
**What is the average cost of acquiring a customer through each acquisition channel?**

In [12]:
# Average Acquisition cost per Channel
avg_acquisitionCost_by_channel = data.groupby('AcquisitionChannel')['AcquisitionCost'].mean().reset_index()

print("Average Acquisition Cost per Channel:")
print(avg_acquisitionCost_by_channel)
# Visualize the average acquisition cost per channel, sorted by revenue in descending order
fig = px.bar(avg_acquisitionCost_by_channel,
             x=avg_acquisitionCost_by_channel.sort_values('AcquisitionCost', ascending=False)['AcquisitionChannel'],  # Sort x-axis by Revenue in descending order
             y=avg_acquisitionCost_by_channel.sort_values('AcquisitionCost', ascending=False)['AcquisitionCost'],  # Ensure y-axis matches the sorted order
             title='Average AcquisitionCost per Acquisition Channel',
             labels={'AcquisitionCost': 'Average AcquisitionCost', 'AcquisitionChannel': 'Acquisition Channel'},
             color='AcquisitionChannel')  # Add color to differentiate channels
fig.update_layout(
    xaxis_title="Acquisition Channel",
    yaxis_title="Average AcquisitionCost"
)

fig.show()

Average Acquisition Cost per Channel:
  AcquisitionChannel  AcquisitionCost
0          Affiliate        40.052336
1    Email Marketing        14.647908
2            Organic         0.000000
3        Paid Search        34.829059
4           Referral         9.853556
5       Social Media        25.092982


- **Organic** has the lowest acquisition cost, which is expected since it typically involves efforts like SEO and content marketing that don't have direct spending per customer.
- **Affiliate** has the highest acquisition cost. This suggests that acquiring customers through affiliate marketing is, on average, the most expensive among the channels considered.
- **Paid Search** also has a relatively high acquisition cost.

Next, I will combine the revenue and acquisition cost metrics into a single visualization to provide a much more clearer and more direct comparison of channel performance.

In [ ]:
# Combine Average Revenue and Acquisition Cost Data
combined_data = pd.merge(average_revenue_by_channel, avg_acquisitionCost_by_channel, on='AcquisitionChannel')

# Melt the dataframe to long format, which is suitable for plotting multiple metrics
combined_data_melted = combined_data.melt(id_vars='AcquisitionChannel',
                                         value_vars=['Revenue', 'AcquisitionCost'],
                                         var_name='Metric',
                                         value_name='Amount')

# Create the combined bar chart
fig = px.bar(combined_data_melted,
             x='AcquisitionChannel',
             y='Amount',
             color='Metric',
             barmode='group',  # Important for side-by-side bars
             title='Average Revenue vs. Acquisition Cost per Channel',
             labels={'Amount': 'Amount', 'AcquisitionChannel': 'Acquisition Channel'},
             color_discrete_map={'Revenue': 'green', 'AcquisitionCost': 'red'}) #Custom colors
fig.show()

**Insights from Average Initial Revenue and Average Acquisition Cost per Acquisition Channel:**

* **Referral:**
    - Highest average initial revenue.
    - Relatively low acquisition cost.

    **Insight:** Referral appears to be a very promising channel. It brings in high-value customers at a reasonable cost, suggesting strong potential for profitability and high CLTV.

* **Email Marketing:**
    - High average initial revenue.
    - Low acquisition cost.
    * **Insight:** Email marketing also looks like a strong channel. It's cost-effective and generates good initial revenue. This indicates efficient acquisition and the potential for repeat business.

* **Affiliate:**
    - Moderate average initial revenue.
    - Highest acquisition cost.
    * **Insight:** Affiliate marketing is the most expensive channel and doesn't necessarily bring in the highest-spending customers. Its high cost warrants a close evaluation of its effectiveness and whether it's delivering sufficient value.

Overall:
1. **Referral and Email Marketing** stand out as high-potential channels due to their combination of strong revenue generation and low acquisition costs.
2. **Costly Channels:** Paid Search and Affiliate are the most expensive channels. Their performance needs to be carefully monitored, and strategies should be explored to reduce costs or increase revenue.
3. **Low Revenue Channels:** Social Media and Organic generate the least revenue. Organic is still very profitable because of the low costs.

## Problem Statement 3
**Is there a correlation between the conversion rate of a channel and the initial revenue of the customers acquired through it?**

Befoe moving on,

**Why is the Correlation Between Conversion Rate and Revenue Important?**

1. **Identifying High-Value Conversion Channels:**
    We want to know if the channels that are good at converting leads into customers also tend to bring in customers who spend more.
    _**If there's a positive correlation, it means that improving the conversion rate of a channel could lead to acquiring more high-value customers, which directly impacts CLTV.**_
2. **Optimizing Marketing Spend:**
If high conversion rates are associated with high revenue, we should prioritize investing in those channels and strategies that improve conversion.

    For example, if we find that "Email Marketing" has a strong positive correlation, we might allocate more resources to email campaigns, A/B testing email content, and improving our email list.

In [16]:
# Correlation between Conversion Rate and Revenue
correlation = data['ConversionRate'].corr(data['Revenue'])
print(f"\nCorrelation between Conversion Rate and Revenue: {correlation:.3f}")

# Visualize the relationship
import plotly.express as px
fig = px.scatter(data, x='ConversionRate', y='Revenue',
                 title='Conversion Rate vs. Revenue',
                 trendline='ols') # Add a trendline to visualize the relationship
fig.show()


Correlation between Conversion Rate and Revenue: 0.394


* A correlation coefficient of `0.39` indicates a moderate positive correlation between conversion rate and revenue.
* This means that **as conversion rates increase, revenue tends to increase**.
* The relationship isn't very strong, but it's also not negligible. There's a discernible tendency for higher conversion rates to be associated with higher revenue.
* In the scatterpot, there is an upward trend in the data points, showing that, in general, as conversion rates rise, so does revenue.
* However, the points are somewhat scattered, indicating that the relationship isn't perfect. There's still a fair amount of variability in revenue that isn't explained by conversion rate alone.

**In the context of our analysis:**
1. This finding suggests that channels with higher conversion rates do have a tendency to bring in customers who spend more initially.
2. While conversion rate is a factor in predicting initial revenue, it's not the only one. Other factors, such as the channel itself, customer demographics, or the offer, also play a significant role.

**However!**

Looking at the overall correlation can mask important differences between channels. It's very possible that the relationship between `conversion rate` and `revenue` varies significantly from one channel to another.

To analyze this, we can calculate the correlation coefficient for each acquisition channel separately.

In [26]:
# Calculate correlation between Conversion Rate and Revenue for each Acquisition Channel
channel_correlations = data.groupby('AcquisitionChannel')[['ConversionRate', 'Revenue']].corr().unstack().iloc[:, 1]
print("\nCorrelation between Conversion Rate and Revenue by Acquisition Channel:")
print(channel_correlations)

# Visualize the relationship for each channel
import plotly.express as px
for channel in data['AcquisitionChannel'].unique():
    channel_data = data[data['AcquisitionChannel'] == channel]
    fig = px.scatter(channel_data, x='ConversionRate', y='Revenue',
                     title=f'Conversion Rate vs. Revenue for {channel}',
                     trendline='ols')
    fig.show()


Correlation between Conversion Rate and Revenue by Acquisition Channel:
AcquisitionChannel
Affiliate         -0.004218
Email Marketing   -0.056972
Organic           -0.011588
Paid Search        0.007159
Referral          -0.003625
Social Media       0.090361
Name: (ConversionRate, Revenue), dtype: float64


In [25]:
data.groupby('AcquisitionChannel')[['ConversionRate', 'Revenue']].corr().unstack().iloc[:, 1]

AcquisitionChannel
Affiliate         -0.004218
Email Marketing   -0.056972
Organic           -0.011588
Paid Search        0.007159
Referral          -0.003625
Social Media       0.090361
Name: (ConversionRate, Revenue), dtype: float64

Wow!

When we break down the correlation between ConversionRate and Revenue by individual acquisition channel, we see that the relationship is much weaker than the overall correlation suggested.

**Interpretation:**

- **Near-Zero Correlation:** For almost every channel, the correlation between conversion rate and revenue is very close to zero. This means that, within each individual channel, there is virtually no linear relationship between how well a channel converts leads and how much revenue those customers generate.
- **Social Media Exception:** Social Media has the highest correlation (0.090), but even this is a very weak positive correlation. It suggests a slight tendency for higher conversion rates to be associated with higher revenue in this channel, but the effect is minimal.

## Problem Statement 4
**What is the initial profitability per customer for each acquisition channel?**

Why this question?

1. **Immediate Channel Performance:** 
    - Initial profitability tells us how much profit we make immediately after acquiring a customer through a specific channel.
    - Some channels might bring in high revenue, but if the acquisition costs are even higher, they're not sustainable in the short term.
2. **Short-Term ROI:** 
    - It provides a quick measure of the return on investment (ROI) for each acquisition channel in the short term.
    - A positive initial profitability means that the channel is generating more revenue than it costs to acquire the customer.



In [ ]:
data['InitialProfit'] = data['Revenue'] - data['AcquisitionCost']
average_profit_by_channel = data.groupby('AcquisitionChannel')['InitialProfit'].mean().reset_index()
average_profit_by_channel_sorted = average_profit_by_channel.sort_values('InitialProfit', ascending=False)

# Visualize the average initial profit per channel, sorted by profit in descending order
fig = px.bar(average_profit_by_channel_sorted,
             x='AcquisitionChannel',
             y='InitialProfit',
             title='Average Initial Profit per Acquisition Channel',
             labels={'InitialProfit': 'Average Initial Profit', 'AcquisitionChannel': 'Acquisition Channel'},
             color='AcquisitionChannel',
             text='InitialProfit')  # Add value labels

# Update the layout to label the x-axis and format the text
fig.update_layout(xaxis_title="Acquisition Channel", yaxis_title="Average Initial Profit")
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')  # Format labels to 2 decimal places

fig.show()


The chart reveals the following key insights:

* **High-Performing Channels:** Referral and email marketing are the most effective channels in terms of initial profitability.
* **Cost-Effective Channels:** Organic acquisition is highly profitable due to its low acquisition costs.
* **Underperforming Channels:** Social Media is the least profitable, followed closely by Affiliate.

_**In the context of CLTV, initial profitability is a starting point. While CLTV focuses on the long-term value of a customer, initial profitability tells us about the immediate value.**_

_**A channel that is initially profitable is in a better position to generate high CLTV, but it's not a guarantee. We need to consider both short-term and long-term perspectives.**_

## Problem Statement 5
**What is the estimated Customer Lifetime Value (CLTV) for customers acquired through different channels assuming an average customer lifetime of 5 years?**

The CLTV is calculated using a simplified formula:

`CLTV = (Average Revenue per Customer) * (Average Customer Lifetime) - (Customer Acquisition Cost)`

In [ ]:
# Sort the data by CLTV in descending order
assumed_lifetime= 5  # Assuming a lifetime of 5 years for CLTV calculation
cltv_data_sorted = cltv_data.sort_values('CLTV', ascending=False)

# Visualize the CLTV with value labels
fig = px.bar(cltv_data_sorted,
             x='AcquisitionChannel',
             y='CLTV',
             title=f'Estimated CLTV per Channel (Lifetime = {assumed_lifetime} years)',
             labels={'CLTV': 'Estimated CLTV'},
             color='AcquisitionChannel',
             text='CLTV')  # Add value labels

# Update the layout to format the text and sort the x-axis
fig.update_layout(xaxis_title="Acquisition Channel", yaxis_title="Estimated CLTV")
fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')  # Format labels to 2 decimal places

fig.show()


**Key Insights from CLTV Analysis (5-Year Lifetime):**
1. **Referral is the most valuable channel:** Customers acquired through referrals have the highest estimated CLTV. This indicates that, over a 5-year period, these customers are expected to generate the most profit for the business. This highlights the importance of encouraging and incentivizing referrals.
2. **Email Marketing is also highly valuable:** Email marketing follows referral as the second most valuable channel. Customers acquired through email marketing are also expected to contribute significant profit over the long term. This reinforces the importance of building and nurturing an email list.
3. **Affiliate and Social Media have lower long-term value:** Affiliate and social media have relatively low CLTV. This implies that customers acquired through these channels are not expected to generate substantial profit over their lifetime. This raises concerns about the cost-effectiveness of these channels.
4. **Organic has the lowest long-term value:** Organic acquisition has the lowest estimated CLTV. While organic traffic is generally cost-free, the lower CLTV suggests that these customers, on average, contribute less to long-term profitability compared to customers from other channels.

## Problem Statement 6
**Which customer acquisition channels are the most profitable based on both initial profitability and estimated long-term value?**

To answer this, we need to consider both the **Average Initial Profit per Acquisition Channel** and the **Estimated CLTV per Channel (Lifetime = 5 years)** analyses.

Combining these insights, we can identify the most profitable channels:

**Referral:** This channel consistently performs exceptionally well in both the short term (highest initial profitability) and the long term (highest CLTV). It attracts high-value customers at a low cost, making it the most profitable channel overall.

**Email Marketing:** Email Marketing is also a highly profitable channel. It has strong initial profitability and the second-highest CLTV. This channel effectively acquires customers who generate good revenue both immediately and over the long term.

_**Therefore, based on our analysis, Referral and Email Marketing are the most profitable customer acquisition channels.**_

## Conclusion: Optimizing Customer Acquisition for Maximum Profitability

This analysis has provided a comprehensive evaluation of customer acquisition channels, combining insights from initial profitability and estimated long-term value (CLTV) to inform strategic decision-making.

**Key Findings:**

1. **Referral and Email Marketing are the most profitable channels.** They consistently deliver high-value customers with strong initial profitability and the highest estimated CLTV. These channels should be prioritized and optimized to maximize customer acquisition and long-term revenue.

2. **Organic acquisition is a cost-effective channel** with high customer volume and zero acquisition costs, resulting in good initial profitability. However, it exhibits the lowest CLTV, indicating lower long-term value per customer.

3. **Paid Search presents a moderate return** with moderate initial profitability and CLTV. While it can drive traffic, its higher acquisition costs require careful optimization to ensure profitability.

4. **Affiliate and Social Media demonstrate the lowest profitability**. Both channels have low initial profitability and CLTV, raising concerns about their cost-effectiveness and long-term value.



## Actionable Recommendations:
Based on these findings, we recommend the following actions:

| Channel          | Action                                                                   | Rationale                                                                                                                              |
| :--------------- | :----------------------------------------------------------------------- | :------------------------------------------------------------------------------------------------------------------------------------- |
| **Referral** | Enhance referral programs with attractive incentives.                     | Referral consistently delivers the highest-value customers with strong profitability.                                                |
|                  | Make the referral process seamless across all platforms.                 | Remove friction to maximize participation and acquisition.                                                                          |
| **Email Marketing** | Invest in building and segmenting email lists.                           | Email marketing provides cost-effective acquisition of high-value customers.                                                           |
|                  | Optimize email campaigns with personalized offers.                       | Improve engagement and conversion rates for this key channel.                                                                       |
| **Affiliate** | Conduct a thorough audit of the affiliate program.                       | Affiliate shows the lowest profitability, indicating potential cost inefficiencies.                                                   |
|                  | Renegotiate commission rates, focusing on performance-based compensation. | Align affiliate incentives with desired outcomes and improve ROI.                                                                   |
| **Social Media** | Re-evaluate the role of social media in customer acquisition.            | Social Media has a low return compared to other channels.                                                                           |
|                  | If used for direct sales, optimize targeting, messaging, and calls to action. | Improve conversion rates if direct sales is the goal.                                                                             |

By implementing these recommendations, stakeholders can optimize their customer acquisition strategies, improve overall profitability, and drive sustainable business growth.